# task #4.2 anonymity

**based on**: Morris Chang, Di Zhuang, G. Dumindu Samaraweera - *Privacy-Preserving Machine Learning* (2023, Manning)

**dataset**: the titanic test file `deepeshkansotia/hantavirus-transmission-and-risk-datasete`.

we anonymizes this dataset using:
- **quasi-identifiers**: `patient_age` and `quarantine_days` (both numerical).
- **sensitive attribute**: `transmission_type` (categorical yes/no) - the most sensitive medical outcome here.

the three privacy models, built on the same recursive-partitioning engine:
- **k-anonymity**: every group of equal quasi-identifiers has at least k records.
- **l-diversity**: on top of k-anonymity, every group has at least l distinct sensitive values.
- **t-closeness**: on top of k-anonymity, every group's sensitive distribution stays
  within distance t of the global distribution.

## 1. import and understand dataset

In [ ]:
import sys
!{sys.executable} -m pip install sklearn-pandas kagglehub --quiet

import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

In [ ]:
import kagglehub, os
path = kagglehub.dataset_download("deepeshkansotia/hantavirus-transmission-and-risk-dataset")
files = os.listdir(path)
df = pd.read_csv(os.path.join(path, [f for f in files if f.endswith(".csv")][0]))
print("shape:", df.shape)
df.head()

100%|██████████| 66.1k/66.1k [00:00<00:00, 21.5MB/s]

Extracting files...
shape: (2000, 19)


,case_id,country,region,report_date,virus_strain,transmission_type,exposure_source,patient_age,gender,symptoms,hospitalization,fatality,recovery_days,temperature_celsius,humidity_percent,rodent_presence_index,quarantine_days,population_density,air_quality_index
0,HV2026_00001,Canada,Lake Rickyside,24-11-2025,Sin Nombre,Rodent-to-Human,Agricultural Exposure,26,Male,"Headache, Vomiting, Fever",Yes,No,22.0,27.0,78.9,7,1,538,67
1,HV2026_00002,Bolivia,East Crystal,22-04-2025,Sin Nombre,Human-to-Human,Home Infestation,41,Female,"Fever, Muscle Pain, Fatigue",Yes,No,41.0,23.2,70.9,3,13,5624,162
2,HV2026_00003,Chile,Rossmouth,21-03-2025,Seoul,Human-to-Human,Home Infestation,39,Female,"Fever, Muscle Pain, Fatigue",No,No,31.0,22.6,38.5,8,17,2095,213
3,HV2026_00004,Argentina,West Ryan,10-02-2025,Sin Nombre,Human-to-Human,Rodent Exposure,49,Male,"Fever, Muscle Pain, Fatigue",No,No,11.0,22.6,87.0,7,8,7478,206
4,HV2026_00005,Chile,Mariaberg,25-03-2025,Dobrava,Rodent-to-Human,Home Infestation,59,Male,"Fever, Cough, Headache",No,No,24.0,33.5,36.2,8,12,4472,132


In [ ]:
for i in df.columns.tolist():
  print("\nCOLUMN NAME\n", i, "\nDATA TYPE\n", df[i].dtype, "\nUNIQUE VALUES\n", df[i].unique(), "\nVALUE COUNTS\n", df[i].value_counts())
  print("________________________________________________")


COLUMN NAME
 case_id 
DATA TYPE
 object 
UNIQUE VALUES
 ['HV2026_00001' 'HV2026_00002' 'HV2026_00003' ... 'HV2026_01998'
 'HV2026_01999' 'HV2026_02000'] 
VALUE COUNTS
 case_id
HV2026_01984    1
HV2026_01983    1
HV2026_01982    1
HV2026_01981    1
HV2026_01980    1
               ..
HV2026_00005    1
HV2026_00004    1
HV2026_00003    1
HV2026_00002    1
HV2026_00001    1
Name: count, Length: 2000, dtype: int64
________________________________________________

COLUMN NAME
 country 
DATA TYPE
 object 
UNIQUE VALUES
 ['Canada' 'Bolivia' 'Chile' 'Argentina' 'Peru' 'Uruguay' 'USA' 'Brazil'
 'Mexico' 'Paraguay'] 
VALUE COUNTS
 country
Bolivia      215
Argentina    211
Canada       208
USA          208
Brazil       207
Uruguay      202
Mexico       196
Paraguay     193
Chile        190
Peru         170
Name: count, dtype: int64
________________________________________________

COLUMN NAME
 region 
DATA TYPE
 object 
UNIQUE VALUES
 ['Lake Rickyside' 'East Crystal' 'Rossmouth' ... 'Cowanbury'


## listing 8.2 - finding the value span
defines which columns are categorical, then computes each column's "span": its range for a
numerical column, or its number of distinct values for a categorical one. the spans tell which column is the widest and therefore the best one to split next.

In [ ]:
# categorical columns of the dataset
categorical = set((
    'case_id', 'country', 'region', 'report_date', 'virus_strain', 'transmission_type',
    'exposure_source', 'gender', 'symptoms', 'hospitalization', 'fatality',
))

for name in categorical:
    df[name] = df[name].astype('category')

def get_spans(df, partition, scale=None):
    spans = {}
    for column in df.columns:
        if column in categorical:
            span = len(df[column][partition].unique())
        else:
            span = df[column][partition].max() - df[column][partition].min()
        if scale is not None:
            span = span / scale[column]
        spans[column] = span
    return spans

full_spans = get_spans(df, df.index)
full_spans

{'case_id': 2000,
 'country': 10,
 'region': 1850,
 'report_date': 490,
 'virus_strain': 5,
 'transmission_type': 2,
 'exposure_source': 6,
 'patient_age': 66,
 'gender': 2,
 'symptoms': 5,
 'hospitalization': 2,
 'fatality': 2,
 'recovery_days': 38.0,
 'temperature_celsius': 43.0,
 'humidity_percent': 60.0,
 'rodent_presence_index': 9,
 'quarantine_days': 21,
 'population_density': 9945,
 'air_quality_index': 280}

## listing 8.3 - partitioning the dataset
splits the data recursively. each partition is cut in half on its widest quasi-identifier (numerical: at the median; categorical: by splitting the value set), and a cut is kept only if **both** halves still satisfy the validity test. here the test is k-anonymity with k=3, so the result is a list of partitions that each contain at least 3 records.

In [ ]:
def split(df, partition, column):
    dfp = df[column][partition]
    if column in categorical:
        values = dfp.unique()
        lv = set(values[:len(values) // 2])
        rv = set(values[len(values) // 2:])
        return dfp.index[dfp.isin(lv)], dfp.index[dfp.isin(rv)]
    else:
        median = dfp.median()
        dfl = dfp.index[dfp < median]
        dfr = dfp.index[dfp >= median]
        return (dfl, dfr)

def is_k_anonymous(df, partition, sensitive_column, k=3):
    if len(partition) < k:
        return False
    return True

def partition_dataset(df, feature_columns, sensitive_column, scale, is_valid):
    finished_partitions = []
    partitions = [df.index]
    while partitions:
        partition = partitions.pop(0)
        spans = get_spans(df[feature_columns], partition, scale)
        for column, span in sorted(spans.items(), key=lambda x: -x[1]):
            lp, rp = split(df, partition, column)
            if not is_valid(df, lp, sensitive_column) or not is_valid(df, rp, sensitive_column):
                continue
            partitions.extend((lp, rp))
            break
        else:
            finished_partitions.append(partition)
    return finished_partitions

In [ ]:
feature_columns = ['patient_age', 'quarantine_days']   # quasi-identifiers
sensitive_column = 'fatality'                          # sensitive attribute
finished_partitions = partition_dataset(df, feature_columns, sensitive_column, full_spans, is_k_anonymous)
print("number of k-anonymous partitions:", len(finished_partitions))

number of k-anonymous partitions: 427


## listing 8.4 - building the anonymized dataset
turns each partition into anonymized rows: numerical quasi-identifiers are replaced by the partition's mean, categorical ones by the set of values they contained and the sensitive attribute is reported as counts per value within the group.

In [ ]:
def agg_categorical_column(series):
    return ",".join(set(series.astype(str)))

def agg_numerical_column(series):
    return series.mean()

def build_anonymized_dataset(df, partitions, feature_columns, sensitive_column, max_partitions=None):
    aggregations = {}
    for column in feature_columns:
        if column in categorical:
            aggregations[column] = agg_categorical_column
        else:
            aggregations[column] = agg_numerical_column
    rows = []
    for i, partition in enumerate(partitions):
        if i % 100 == 1:
            print("Finished {} partitions...".format(i))
        if max_partitions is not None and i > max_partitions:
            break
        grouped_columns = df.loc[partition].agg(aggregations)
        sensitive_counts = df.loc[partition].groupby(sensitive_column, observed=True).agg(
            {sensitive_column: 'count'})
        values = grouped_columns.to_dict()
        for sensitive_value, count in sensitive_counts[sensitive_column].items():
            if count == 0:
                continue
            values.update({sensitive_column: sensitive_value, 'count': count})
            rows.append(values.copy())
    return pd.DataFrame(rows)

In [ ]:
dfn = build_anonymized_dataset(df, finished_partitions, feature_columns, sensitive_column)
print("k-anonymized dataset shape:", dfn.shape)
dfn.head()

Finished 1 partitions...
Finished 101 partitions...
Finished 201 partitions...
Finished 301 partitions...
Finished 401 partitions...
k-anonymized dataset shape: (563, 4)


,patient_age,quarantine_days,fatality,count
0,30.285714,0.142857,No,7
1,33.857143,0.428571,No,7
2,36.090909,0.272727,No,10
3,36.090909,0.272727,Yes,1
4,30.428571,11.428571,No,7


## listing 8.5 - l-diversity (l=2)
re-runs the partitioning with a stricter test: each group must satisfy k-anonymity **and**
contain at least l=2 distinct `fatality` values. groups that would have been all-"No" are no
longer allowed, so the engine produces fewer, more mixed groups.

In [ ]:
def diversity(df, partition, column):
    return len(df[column][partition].unique())

def is_l_diverse(df, partition, sensitive_column, l=2):
    return diversity(df, partition, sensitive_column) >= l

finished_l_diverse_partitions = partition_dataset(
    df, feature_columns, sensitive_column, full_spans,
    lambda *args: is_k_anonymous(*args) and is_l_diverse(*args))

print("number of l-diverse partitions:", len(finished_l_diverse_partitions))

column_x, column_y = feature_columns[:2]
dfl = build_anonymized_dataset(df, finished_l_diverse_partitions, feature_columns, sensitive_column)
dfl.sort_values([column_x, column_y, sensitive_column]).head(10)

number of l-diverse partitions: 127
Finished 1 partitions...
Finished 101 partitions...


,patient_age,quarantine_days,fatality,count
152,12.000000,9.000000,No,3
153,12.000000,9.000000,Yes,1
50,12.545455,3.181818,No,10
51,12.545455,3.181818,Yes,1
86,12.818182,17.545455,No,10
87,12.818182,17.545455,Yes,1
90,12.909091,20.090909,No,9
91,12.909091,20.090909,Yes,2
168,13.200000,15.200000,No,4
169,13.200000,15.200000,Yes,1


## listing 8.6 - checking the global frequencies
computes the global distribution of the sensitive attribute over the whole dataset. these frequencies are the reference that t-closeness compares each group against.

In [ ]:
global_freqs = {}
total_count = float(len(df))
group_counts = df.groupby(sensitive_column, observed=True)[sensitive_column].agg('count')
for value, count in group_counts.to_dict().items():
    p = count / total_count
    global_freqs[value] = p * 100
print(global_freqs, "in percentage (%)")

{'No': 92.25, 'Yes': 7.75} in percentage (%)


## listing 8.7 - t-closeness (t=0.2)
re-runs the partitioning so that, in addition to k-anonymity, each group's `fatality` distribution differs from the global distribution by at most t=0.2.

In [ ]:
t_sensitive_column = 'transmission_type'   # balanced sensitive attribute for t-closeness

global_freqs = {}
total_count = float(len(df))
group_counts = df.groupby(t_sensitive_column, observed=True)[t_sensitive_column].agg('count')
for value, count in group_counts.to_dict().items():
    p = count / total_count
    global_freqs[value] = p
print(global_freqs)

{'Human-to-Human': 0.5005, 'Rodent-to-Human': 0.4995}


In [ ]:
def t_closeness(df, partition, column, global_freqs):
    total_count = float(len(partition))
    d_max = None
    group_counts = df.loc[partition].groupby(column, observed=True)[column].agg('count')
    for value, count in group_counts.to_dict().items():
        p = count / total_count
        d = abs(p - global_freqs[value])
        if d_max is None or d > d_max:
            d_max = d
    return d_max

def is_t_close(df, partition, sensitive_column, global_freqs, p=0.2):
    if not sensitive_column in categorical:
        raise ValueError("this method only works for categorical values")
    return t_closeness(df, partition, sensitive_column, global_freqs) <= p

In [ ]:
finished_t_close_partitions = partition_dataset(
    df, feature_columns, t_sensitive_column, full_spans,
    lambda *args: is_k_anonymous(*args) and is_t_close(*args, global_freqs))

print("number of t-close partitions:", len(finished_t_close_partitions))

dft = build_anonymized_dataset(df, finished_t_close_partitions, feature_columns, t_sensitive_column)
dft.sort_values([column_x, column_y, t_sensitive_column]).head(10)

number of t-close partitions: 233
Finished 1 partitions...
Finished 101 partitions...
Finished 201 partitions...


,patient_age,quarantine_days,transmission_type,count
266,12.000000,5.666667,Human-to-Human,2
267,12.000000,5.666667,Rodent-to-Human,1
166,12.000000,20.000000,Human-to-Human,3
167,12.000000,20.000000,Rodent-to-Human,2
124,12.500000,3.625000,Human-to-Human,3
125,12.500000,3.625000,Rodent-to-Human,5
122,12.666667,2.000000,Human-to-Human,2
123,12.666667,2.000000,Rodent-to-Human,1
54,12.818182,17.545455,Human-to-Human,7
55,12.818182,17.545455,Rodent-to-Human,4
